# Practice for Jyoti in the whole RAG Data Ingestion Pipeline code

In [1]:
# Step 1 - Use Directory Loaders and PDF Loaders to read the files and get your Document Object

from langchain_core.documents import Document
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, CSVLoader
import os


mypath_pdf = "../data/pdf_files"
mypath_csv = "../data/csv_files"

mypath_pdf = os.path.abspath(mypath_pdf)
mypath_csv = os.path.abspath(mypath_csv)

pdf_directory_loader = DirectoryLoader(mypath_pdf, glob="**/*.pdf", loader_cls=PyPDFLoader)
csv_directory_loader = DirectoryLoader(mypath_csv, glob="**/*.csv", loader_cls=CSVLoader)
pdf_documents = pdf_directory_loader.load()
csv_documents = csv_directory_loader.load()

print ("Loaded all documents ",len(pdf_documents) + len(csv_documents))



C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11256\964502540.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, CSVLoader
e:\Users\Jyoti\Projects\RAGPart1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded all documents  45


In [2]:
# Step 2 - Split the documents into chunks and get your Document Object

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
#chunks = text_splitter.split_documents(pdf_documents)
chunks = text_splitter.split_documents(csv_documents)

print ("Total chunks created ",len(chunks))




Total chunks created  64


In [3]:
# Step 3 - Convert text to embeddings and add the documents and embeddings to the vector store
import uuid
import numpy as np
from sentence_transformers import SentenceTransformer 
from typing import List


class EmbeddingManager:
    def __init__(self, model_name):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """
        Load the embedding model based on the specified model name.
        """
        try :
            self.embedding_model = SentenceTransformer(self.model_name)
            print ("Successfully loaded model ", self.model_name)
            print ("Model dimension: ", self.embedding_model.get_embedding_dimension())

        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise


    
    def generate_embeddings(self, texts : List[str]) -> list[np.ndarray]:
        """
        Generate embeddings for a list of texts using the specified embedding model.
        
        Args:
            Texts : A list of String objects whose content will be converted into embeddings.
        
        Returns:
            A list of numpy arrays, where each array is the embedding of the corresponding text.
        """
        try:
            #texts = []
            #embeddings = []
        
            #texts = [doc.page_content for doc in Documents]
            embeddings = self.embedding_model.encode(texts,show_progress_bar=True)
            print ("Embeddings done ", embeddings.size)
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise
        
      

embedding_manager = EmbeddingManager(model_name="all-MiniLM-L6-v2")
# Extract the text strings from the LangChain document objects
texts_to_embed = [doc.page_content for doc in chunks]

# Pass the list of strings to your manager
embeddings = embedding_manager.generate_embeddings(texts_to_embed)

#embeddings = embedding_manager.generate_embeddings(chunks)  # Generate embeddings for the chunks
print ("Embeddings generated for chunks ", len(chunks))


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 893.32it/s]


Successfully loaded model  all-MiniLM-L6-v2
Model dimension:  384


Batches: 100%|██████████| 2/2 [00:05<00:00,  2.95s/it]

Embeddings done  24576
Embeddings generated for chunks  64


In [5]:
import chromadb
from chromadb.config import Settings
from typing import List, Dict, Tuple, Any
import uuid
import os


class VectorStoreDB:
    def __init__(self, persist_directory: str, collection_name: str):
       
        self.persistent_directory =  persist_directory
        self.client = None
        self.collection = None
        self.collection_name = collection_name
        self.create_vector_store (self.persistent_directory,self.collection_name)

    def create_vector_store(self,persist_directory: str, collection_name: str):
        """
        Create a vector store with the specified persistence directory and collection name.
        """
        try :
            self.persistent_directory = os.path.abspath(persist_directory)



            self.client = chromadb.PersistentClient(path=self.persistent_directory,
                                                     settings=Settings(anonymized_telemetry=False))
            


            self.collection = self.client.get_or_create_collection(
                                name=collection_name,
                                metadata={"description": "Collection for storing CSV document embeddings"}
                                )
            print ("successfully created vector store with collection name ", collection_name)

        except Exception as e:
            print(f"Error creating vector store: {e}")
            raise


    def add_documents(self, documents: list[Document], embeddings: list[np.ndarray]):
        ###
        #Add or update documents and their corresponding embeddings in the vector store idempotently.
        ##
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents must match the number of embeddings.")
        
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Safe extraction whether it's a Document object or a Dict
            if hasattr(doc, "metadata"):
                metadata = dict(doc.metadata)
                page_content = doc.page_content
            else:
                metadata = dict(doc.get("metadata", {}))
                page_content = doc.get("page_content", "")                  
            
            # --- 1. GENERATE DETERMINISTIC UNIQUE ID ---
            # Get the source filename or fall back to a default identifier
            source_file = metadata.get("source", "unknown_source")
            # Clean up the file path string to create a clean alphanumeric ID token
            clean_source = os.path.basename(source_file).replace('.', '_').replace(' ', '_')
            
            # This ID stays identical across script restarts for the exact same file chunks
            doc_id = f"doc_{clean_source}_{i}"  
            
            ids.append(doc_id)
            metadata["doc_index"] = i
            metadata["content_length"] = len(page_content)
            
            metadatas.append(metadata)
            documents_text.append(page_content)
            embeddings_list.append(embedding.tolist())  # Convert numpy array to list for storage
        
        # --- 2. USE UPSERT INSTEAD OF ADD ---
        # upsert() will insert if the ID is new, or update/overwrite if the ID already exists
        self.collection.upsert(
            ids=ids,
            metadatas=metadatas,
            documents=documents_text,
            embeddings=embeddings_list
        )
        print(f"Successfully indexed/upserted documents in collection '{self.collection_name}'. Current total count: {self.collection.count()}")

vectorstoredb = VectorStoreDB(persist_directory="../data/vectorstore",collection_name="csv_collection")
print ("Vector store created successfully with collection name ", vectorstoredb.collection.name)

vectorstoredb.add_documents(chunks, embeddings)  # Add documents and embeddings to the vector store



successfully created vector store with collection name  csv_collection
Vector store created successfully with collection name  csv_collection
Successfully indexed/upserted documents in collection 'csv_collection'. Current total count: 64


#### RAG Retriever from Vector Store

In [ ]:
class RAGRetriver:

    """ Handles query based retrieval
    """

    def __init__(self, vector_store: VectorStoreDB, embedding_manager : EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve (self, query: str, top_k : int = 5, score_thereshold: float = 0.0) -> List[Dict[str,Any]]:
        # Retrieve relevant documents for a query
        #
        # Args: 
        #       query - The search string
        #       top_k - number of top results to return
        #       score_threshold - Minimum similarity score threshold
        # Returns :
        #       List of retrieved documents and metadata

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k} and Score Threshold: {score_thereshold}")

        # Generating embeddings
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
       # query_embedding = self.embedding_model.encode(query, convert_to_numpy=True)
        print (f"Query embedding done : {query_embedding}")

        # search in the vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results = top_k
            )

            retrieved_docs = []

            if (results['documents'] and results['documents'][0] ):
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]


                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance
                    if (similarity_score >= score_thereshold):
                        retrieved_docs.append({
                                'id': doc_id,
                                'document': document,
                                'metadata': metadata,
                                'similarity_score': similarity_score,
                                'distance': distance,
                                'rank' : i+1
                        })
                print (f"Retrieved {len(retrieved_docs)} documents after filtering")
            else:
                print ("No documents found")

            return retrieved_docs

        except Exception as e:
            print (f"Error during retrieval {e}")
            return []


ragRetriver = RAGRetriver(vectorstoredb,embedding_manager)
ragRetriver.retrieve("How do I set up VPN access on my laptop")


        
        

    




Retrieving documents for query: 'How do I set up VPN access on my laptop'
Top K: 5 and Score Threshold: 0.0


Batches: 100%|██████████| 1/1 [00:00<00:00, 21.59it/s]

Embeddings done  384
Query embedding done : [-2.74052490e-02  5.91561422e-02 -1.47518469e-02 -4.28726599e-02
 -4.31049131e-02  1.02593213e-01 -1.25444541e-03  3.64095792e-02
  3.89476120e-02 -3.71900350e-02  3.40905897e-02  3.92689593e-02
 -2.43275911e-02  3.62427137e-03  1.02700805e-02 -7.02393521e-03
  1.02097793e-02 -5.70472293e-02  5.50236218e-02 -7.04629272e-02
  5.42766564e-02 -5.36682196e-02 -1.41530968e-02 -9.61130857e-02
  4.44939546e-02 -6.02972619e-02  5.88234924e-02  2.20265612e-02
 -1.50550259e-02  1.08782418e-01 -1.42552366e-03 -7.22382367e-02
 -3.10196690e-02  2.86233854e-02 -4.93729375e-02 -8.22614059e-02
 -5.69047593e-03  7.36281723e-02  2.79050525e-02  3.55021060e-02
 -2.88705193e-02  2.26547080e-03  5.72106577e-02 -7.54019246e-03
  1.22385986e-01  1.45561656e-03  6.26626909e-02  2.36071218e-02
  6.58548027e-02 -5.93527965e-02  5.86958379e-02 -4.54848744e-02
  1.51432808e-02 -6.36608973e-02 -8.29328075e-02  6.67444110e-05
  4.64403769e-03  3.85963880e-02  2.45901365e-

[{'id': 'doc_rag_sample_qas_from_kis_csv_16',
  'document': '1. First, ensure your laptop meets the company\'s minimum system requirements for VPN connectivity.\n2. Go to the company\'s software portal and download the VPN client software.\n3. Run the installer and follow the prompts to install the software.\n4. Once installed, launch the VPN client software and create a new connection by selecting "Company VPN" as the connection type.\n5. Enter the company\'s VPN server address: `vpn.company.com`, and your company login credentials (username and password).\n6. Save the connection settings and click on the "Connect" button to establish the VPN connection.\n7. You may be prompted to authenticate with two-factor authentication (2FA), so follow the prompts to complete 2FA.\n8. Once connected, verify that you can access company resources by opening a web browser and navigating to `company.com`. You should also verify that your device\'s IP address has changed to a company-assigned IP addre